<a href="https://colab.research.google.com/github/sandyasss15-ai/ai-mentor-portfolio/blob/main/Day9_LabA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
!pip install -q langgraph langchain-google-genai langchain-community duckduckgo-search

import os, getpass
if 'GEMINI_API_KEY' not in os.environ:
    os.environ['GEMINI_API_KEY'] = getpass.getpass('Gemini API key: ')

In [8]:
!pip install -q ddgs
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun

@tool
def web_search(query: str) -> str:
    """Search the web for up-to-date information.
    Use when the question requires current events, recent facts, or
    information not in static training knowledge."""
    return DuckDuckGoSearchRun().run(query)

# Test the tool directly
print(web_search.invoke({'query': 'TCS hiring 2026'})[:400])

How do you create remarkable change? By hiring, celebrating and nurturing the best people-from all walks of life. We’re here to help! Tell us what you’re looking for and we’ll get you connected to the right people. Discover the latest TCS hiring 2026 updates, including TCS recruitment 2026, upcoming TCS jobs 2026, and all new TCS vacancies 2026 across India. Find complete details on TCS freshers h


In [9]:
from langgraph.prebuilt import create_react_agent
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash')
agent = create_react_agent(llm, tools=[web_search])

print('Agent created.')

Agent created.


/tmp/ipykernel_5037/1070485237.py:5: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools=[web_search])


In [11]:
result = agent.invoke({
    'messages': [('user', "What is TCS's 2026 hiring quota?")]
})

# Print every message in the conversation
for i, m in enumerate(result['messages']):
    print(f'\n[{i}] {type(m).__name__}')
    if hasattr(m, 'content'):
        print(f'    Content: {str(m.content)[:300]}')
    if hasattr(m, 'tool_calls') and m.tool_calls:
        print(f'    Tool calls: {m.tool_calls}')


[0] HumanMessage
    Content: What is TCS's 2026 hiring quota?

[1] AIMessage
    Content: 
    Tool calls: [{'name': 'web_search', 'args': {'query': 'TCS 2026 hiring quota'}, 'id': 'd12f2f94-f71a-4b6f-9c39-e40476e3a622', 'type': 'tool_call'}]

[2] ToolMessage
    Content: India’s largest IT services firm, Tata Consultancy Services (TCS), has made 25,000 job offers to fresh graduates for FY2026-27, signaling a cautious yet continued commitment to campus hiring. Tata Consultancy Services (TCS) (BSE: 532540, NSE: TCS) is a digital transformation and technology partner o

[3] AIMessage
    Content: [{'type': 'text', 'text': 'TCS has made 25,000 job offers to fresh graduates for FY2026-27. The company also aims to onboard around 40,000 freshers annually. There are also reports of TCS hiring over 44,000 freshers in the Financial Year 2026.', 'extras': {'signature': 'CqYMAQw51sc4ZGVlgDzRZa7I0g9Ea


In [12]:
# Pass a question that should fail the tool
result = agent.invoke({
    'messages': [('user', 'Search this URL and tell me what it says: https://this-domain-does-not-exist-12345.example.com/jd')]
})

# Watch how the agent recovers
for i, m in enumerate(result['messages']):
    print(f'\n[{i}] {type(m).__name__}')
    if hasattr(m, 'content'):
        print(f'    {str(m.content)[:300]}')


[0] HumanMessage
    Search this URL and tell me what it says: https://this-domain-does-not-exist-12345.example.com/jd

[1] AIMessage
    [{'type': 'text', 'text': 'I am sorry, I cannot directly access the content of a URL. Additionally, the domain in the URL you provided appears to be invalid.', 'extras': {'signature': 'CoAHAQw51sevNSOReGZho+Qu2ziMzNB8F3RGzrg03SzK7J3AZoN2rMjnBMGNp0CicwYlCiW3mo4F4H3DLULGH+2ytJnF00LQvp55Ia/5yUw7dDKfz3S


In [13]:
import requests
from bs4 import BeautifulSoup
from langchain_core.tools import tool

@tool
def jd_fetcher(url: str) -> str:
    """Fetch a job description from a URL and return clean plain text.
    Use when the user provides a job posting URL and you need the JD content.
    Returns first 4000 characters of the cleaned page text."""
    try:
        r = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'}, timeout=10)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, 'html.parser')
        for tag in soup(['script', 'style']):
            tag.decompose()
        return soup.get_text(separator='\n', strip=True)[:4000]
    except Exception as e:
        return f'ERROR: failed to fetch URL — {e}'

In [14]:
@tool
def skills_gap(student_skills: str, must_have_skills: str) -> str:
    """Compare a student's skills (comma-separated) to a job's must-have skills (comma-separated).
    Returns missing skills, comma-separated, or 'none' if student has all.
    Use when the user provides a student profile and a JD's required skills."""
    a = set(s.strip().lower() for s in student_skills.split(',') if s.strip())
    b = set(s.strip().lower() for s in must_have_skills.split(',') if s.strip())
    missing = sorted(b - a)
    return ', '.join(missing) if missing else 'none'

# Test
print(skills_gap.invoke({
    'student_skills': 'Python, Java, SQL',
    'must_have_skills': 'Python, Java, SQL, Spring Boot, AWS',
}))
# Expected: 'aws, spring boot'

aws, spring boot


In [15]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash')

@tool
def answer_scorer(question: str, answer: str) -> str:
    """Score a student's answer to a placement interview question, 1-10, with one-line rationale.
    Use when evaluating how well a student answered a specific interview question.
    Returns format: 'Score: X/10. Rationale: <reason>'."""
    prompt = (f'Score this placement interview answer 1-10 with one-line rationale.\n'
              f'Question: {question}\n'
              f'Answer: {answer}')
    return llm.invoke(prompt).content

# Test
print(answer_scorer.invoke({
    'question': 'Why TCS Digital?',
    'answer': 'Because TCS is big and they pay well.',
}))
# Expected: low score (~3-4/10), rationale about lack of specificity / cultural fit

**Score: 3/10**

**Rationale:** Highly generic and transactional, demonstrating no specific interest in TCS Digital's work or mission beyond compensation.


In [16]:
import os

# Create data directory if it doesn't exist
if not os.path.exists('data'):
    os.makedirs('data')

# Sample student profiles data
student_profiles_data = [
  {
    "name": "Alice Smith",
    "branch": "Computer Science",
    "cgpa": 8.5,
    "skills": ["Python", "Machine Learning", "SQL", "Data Structures"],
    "target_company": "Google"
  },
  {
    "name": "Bob Johnson",
    "branch": "Electronics",
    "cgpa": 7.8,
    "skills": ["C++", "Embedded Systems", "Robotics"],
    "target_company": "Tesla"
  }
]

# Write the data to a JSON file
with open('./data/student_profiles.json', 'w') as f:
    json.dump(student_profiles_data, f, indent=2)

print("Created './data/student_profiles.json' with sample data.")

from langgraph.prebuilt import create_react_agent

tools = [jd_fetcher, skills_gap, answer_scorer]
agent = create_react_agent(llm, tools=tools)
print(f'Agent created with {len(tools)} tools.')

Created './data/student_profiles.json' with sample data.
Agent created with 3 tools.


/tmp/ipykernel_5037/707072847.py:34: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools=tools)


In [17]:
import json, pathlib
from langgraph.prebuilt import create_react_agent

# Re-define agent, assuming llm and the tool functions are globally available
tools = [jd_fetcher, skills_gap, answer_scorer]
agent = create_react_agent(llm, tools=tools)

profiles = json.loads(pathlib.Path('../content/sample_data/student_profiles.json').read_text())

for i, p in enumerate(profiles):
    print(f'\n{"="*70}')
    print(f'Student {i+1}: {p["name"]} — {p["branch"]} CGPA {p["cgpa"]} → {p["target_company"]}')
    print(f'{"="*70}')

    msg = (f"I am {p['name']}, B.Tech {p['branch']} CGPA {p['cgpa']}, "
           f"skills: {', '.join(p['skills'])}. Target: {p['target_company']}. "
           f"Plan 3 mock interview questions for me, score one of my sample answers, "
           f"and tell me what skills I need to add to be a strong fit.")

    result = agent.invoke({'messages': [('user', msg)]}, config={'recursion_limit': 10})

    for j, m in enumerate(result['messages']):
        print(f'\n  [{j}] {type(m).__name__}')
        if hasattr(m, 'content') and m.content:
            print(f'      {str(m.content)[:300]}')
        if hasattr(m, 'tool_calls') and m.tool_calls:
            for tc in m.tool_calls:
                print(f'      → tool_call: {tc.get("name")}({tc.get("args")})')

/tmp/ipykernel_5037/1862008609.py:6: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools=tools)



Student 1: Ravi Kumar — CSE CGPA 8.2 → TCS Digital

  [0] HumanMessage
      I am Ravi Kumar, B.Tech CSE CGPA 8.2, skills: Python, Java, SQL, Git. Target: TCS Digital. Plan 3 mock interview questions for me, score one of my sample answers, and tell me what skills I need to add to be a strong fit.

  [1] AIMessage
      [{'type': 'text', 'text': 'Hello Ravi! That\'s a great goal. Let\'s get you prepared for TCS Digital.\n\nHere are 3 mock interview questions for you:\n\n1.  **Technical:** Explain the concept of Object-Oriented Programming (OOP) and describe its four main pillars with examples in Java or Python.\n2.

Student 2: Sneha Reddy — ECE CGPA 7.6 → Cognizant

  [0] HumanMessage
      I am Sneha Reddy, B.Tech ECE CGPA 7.6, skills: C++, Python, MATLAB, Verilog. Target: Cognizant. Plan 3 mock interview questions for me, score one of my sample answers, and tell me what skills I need to add to be a strong fit.

  [1] AIMessage
      [{'type': 'text', 'text': 'Hello Sneha, it\'s great

In [18]:
# Pass a bad URL — see how agent recovers
result = agent.invoke({
    'messages': [('user', 'Fetch this JD and tell me the must-have skills: '
                          'https://this-does-not-exist-99999.example.com/jd')]
}, config={'recursion_limit': 5})

print('Failure recovery trace:')
for j, m in enumerate(result['messages']):
    print(f'\n[{j}] {type(m).__name__}')
    if hasattr(m, 'content') and m.content:
        print(f'    {str(m.content)[:300]}')
    if hasattr(m, 'tool_calls') and m.tool_calls:
        for tc in m.tool_calls:
            print(f'    → {tc.get("name")}({tc.get("args")})')

Failure recovery trace:

[0] HumanMessage
    Fetch this JD and tell me the must-have skills: https://this-does-not-exist-99999.example.com/jd

[1] AIMessage
    → jd_fetcher({'url': 'https://this-does-not-exist-99999.example.com/jd'})

[2] ToolMessage
    ERROR: failed to fetch URL — HTTPSConnectionPool(host='this-does-not-exist-99999.example.com', port=443): Max retries exceeded with url: /jd (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7eb45b8af080>: Failed to resolve 'this-does-not-exist-99999.example.com' ([Errn

[3] AIMessage
    [{'type': 'text', 'text': "I wasn't able to fetch the JD from the URL you provided. Please check the link and try again with a valid URL.", 'extras': {'signature': 'CrACAQw51sdGb38FVjdL6qxtGRaZ7eilhPGkH+zWlOAnkS3j+Dm3hOBkYXtLjr5/PIfhbCFwr/poea5if5GlsBzdvaXDKuG4POgUfPuC40nNfB4xSluy9HMp6jEYRnN025a5Rtd
